#Integrated Silver Dataset

📌 Goal

Create one integrated Silver dataset by combining:

Silver grid events
Silver asset reference
Region context

Final output table:

## Why This Integrated Dataset is Useful

Instead of using many separate tables, this dataset gives one centralized operational view.

Benefits:
Easier analytics
Faster reporting
Better joins for Gold layer
Regional operational analysis
Asset + event visibility in one table

In [0]:
spark.sql("DROP TABLE IF EXISTS vattenfall_dev.refined.silver_regional_operations_base")

In [0]:
from pyspark.sql import functions as F

grid_events = spark.table(
    "vattenfall_dev.refined.silver_grid_events"
)

asset_reference = spark.table(
    "vattenfall_dev.raw.ref_asset_reference"
)

region_context = spark.table(
    "vattenfall_dev.refined.silver_region"
)

In [0]:
asset_reference_clean = asset_reference.drop("region")

events_assets = grid_events.join(
    asset_reference_clean,
    on="asset_id",
    how="left"
)

print("Asset join completed")

In [0]:
region_context_clean = region_context.select(
    "region_key",
    "country_code",
    "operating_zone",
    "country_region_key"
)

silver_integrated = events_assets.join(
    region_context_clean,
    events_assets["region"] == region_context_clean["region_key"],
    "left"
)

print("Region join fixed (no duplicates)")

In [0]:
silver_integrated = silver_integrated \
    .withColumn(
        "event_year",
        F.year(F.col("event_ts"))
    ) \
    .withColumn(
        "is_critical_event",
        F.when(
            F.col("severity") == "HIGH",
            True
        ).otherwise(False)
    )

print("Business columns added")

In [0]:
spark.table("vattenfall_dev.raw.ref_asset_reference").printSchema()
spark.table("vattenfall_dev.refined.silver_region").printSchema()
spark.table("vattenfall_dev.refined.silver_grid_events").printSchema()

In [0]:
silver_integrated.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "vattenfall_dev.refined.silver_regional_operations_base"
    )

print("✅ Silver integrated dataset created successfully")

In [0]:
df = spark.table(
    "vattenfall_dev.refined.silver_regional_operations_base"
)

display(df.limit(20))

# Correct Join (Fix for NULL issue)

In this step, we join the Silver grid events with the Silver asset reference table.

We use only `asset_id` as the join key because it is the unique identifier for assets.

This avoids mismatches caused by region differences and ensures that asset details like `asset_name` and `asset_type` are correctly populated.

The result is a clean, reliable integrated dataset.

In [0]:
df_base = df_events.alias("e").join(
    df_assets.alias("a"),
    F.col("e.asset_id") == F.col("a.asset_id"),
    "left"
)

In [0]:
df_base = df_base.select(
    "e.event_id",
    "e.event_ts",
    "e.event_day",
    "e.region",
    "e.asset_id",
    "e.event_type",
    "e.severity",
    "e.severity_band",
    "e.duration_minutes",
    "a.asset_name",
    "a.asset_type",
    "e.source_system",
    "e.last_updated_ts",
    "e.ingestion_ts"
)

In [0]:
display(df_base)
df_base.printSchema()
print("Row count:", df_base.count())

In [0]:
df_base.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{refined_schema}.silver_regional_operations_base")

In [0]:
display(
    spark.table(f"{catalog}.{refined_schema}.silver_regional_operations_base")
)

In [0]:
# Transform Bronze asset reference to Silver
bronze_asset_df = spark.table("vattenfall_dev.raw.ref_asset_reference")

# Optional: trim string columns
for col_name, dtype in bronze_asset_df.dtypes:
    if dtype == "string":
        bronze_asset_df = bronze_asset_df.withColumn(col_name, F.trim(F.col(col_name)))

# Write to Silver (refined) schema
bronze_asset_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("vattenfall_dev.refined.silver_asset_reference")

print("Silver asset reference table created")